In [ ]:
import requests
import pandas as pd
from google.colab import files

# ====== CONFIG ======
FRED_API_KEY = "YOUR_API_KEY"
START_DATE = "2018-01-01"
END_DATE   = "2025-12-31"

FRED_SERIES = {
    "FED_RATE": "FEDFUNDS",
    "INFLATION_CPI": "CPIAUCSL",
    "CONSUMER_CONFIDENCE": "UMCSENT",
    "SP500": "SP500",
    "VIX": "VIXCLS",
    "US10Y": "DGS10"
}

FACT_CSV = "fact_macro_daily.csv"
DIM_DATE_CSV = "dim_date.csv"
RESAMPLE_TO_BUSINESS_DAYS = True
# =====================

def validate_key(api_key: str):
    bad = (api_key is None) or (len(api_key.strip()) < 10) or ("PUT" in api_key.upper()) or (" " in api_key)
    if bad:
        raise ValueError("FRED_API_KEY לא תקין. שימי מפתח אמיתי (בלי רווחים).")

def fetch_fred_series(series_id: str, api_key: str, start: str, end: str) -> pd.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key.strip(),
        "file_type": "json",
        "observation_start": start,
        "observation_end": end
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    obs = r.json().get("observations", [])
    df = pd.DataFrame(obs)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df.sort_values("date")

def build_dim_date(start_date: str, end_date: str) -> pd.DataFrame:
    d = pd.date_range(start=start_date, end=end_date, freq="D")
    dim = pd.DataFrame({"date": d})
    dim["year"] = dim["date"].dt.year
    dim["month"] = dim["date"].dt.month
    dim["month_name"] = dim["date"].dt.strftime("%B")
    dim["quarter"] = "Q" + dim["date"].dt.quarter.astype(str)
    dim["day_of_week"] = dim["date"].dt.day_name()
    return dim

def main():
    validate_key(FRED_API_KEY)

    frames = []
    for col_name, series_id in FRED_SERIES.items():
        s = fetch_fred_series(series_id, FRED_API_KEY, START_DATE, END_DATE)
        s = s.rename(columns={"value": col_name}).set_index("date")
        frames.append(s)

    macro = pd.concat(frames, axis=1).sort_index()

    if RESAMPLE_TO_BUSINESS_DAYS:
        macro = macro.asfreq("B").ffill()
    else:
        macro = macro.ffill()

    fact_macro = macro.reset_index().rename(columns={"index": "date"})
    fact_macro.to_csv(FACT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {FACT_CSV} | rows={len(fact_macro)} | cols={len(fact_macro.columns)}")

    dim_date = build_dim_date(START_DATE, END_DATE)
    dim_date.to_csv(DIM_DATE_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {DIM_DATE_CSV} | rows={len(dim_date)}")

    # הורדה ב-Colab
    files.download(FACT_CSV)
    files.download(DIM_DATE_CSV)

if __name__ == "__main__":
    main()


Saved: fact_macro_daily.csv | rows=2088 | cols=7
Saved: dim_date.csv | rows=2922


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>